In [0]:
%pip install requests

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA notebook_breweries;

In [0]:
%skip
import requests
url = "https://api.openbrewerydb.org/v1/breweries/meta"
meta = requests.get(url).json()

display(meta)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("id",             StringType(), True),
    StructField("name",           StringType(), True),
    StructField("brewery_type",   StringType(), True),
    StructField("address_1",      StringType(), True),
    StructField("address_2",      StringType(), True),
    StructField("address_3",      StringType(), True),
    StructField("city",           StringType(), True),
    StructField("state_province", StringType(), True),
    StructField("postal_code",    StringType(), True),
    StructField("country",        StringType(), True),
    StructField("longitude",      StringType(), True),
    StructField("latitude",       StringType(), True),
    StructField("phone",          StringType(), True),
    StructField("website_url",    StringType(), True),
    StructField("state",          StringType(), True),
    StructField("street",         StringType(), True),
])

In [0]:
import requests

per_page = 10 

# Quanti record sono già presenti?
if spark.catalog.tableExists("bronze_breweries"):

    total_existing = spark.table("bronze_breweries").count()

else: 
    total_existing = 0

# Calcolo della pagina da prendere
current_page = (total_existing // per_page) + 1 # considera che la page non è che ha un valore fisso, quindi può essere variabile e ha un valore max di 200 elementi

# Chiamata API
url = "https://api.openbrewerydb.org/v1/breweries"

params = {
    "per_page" : per_page,
    "page" : current_page
}

response = requests.get(url, params = params)

data = response.json()

if data:
    df = spark.createDataFrame(data, schema = schema)

In [0]:
%skip
# nel caso di prendere solo le prime 200 tramite chiamata API
import requests
import json
from pyspark.sql import SparkSession

url = "https://api.openbrewerydb.org/v1/breweries?per_page=200"

response = requests.get(url)
data = response.json()

spark = SparkSession.builder.appName("Breweries").getOrCreate()
df_raw = spark.createDataFrame(data)

In [0]:
# Save as a managed Delta table in Unity Catalog (replace with your catalog/schema if needed)
catalog = "workspace"
schema = "notebook_breweries"
table_name = "bronze_breweries"

df_raw.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.{table_name}")